In [5]:
# === run_simulation.py ===
import numpy as np
from init_state import generate_initial_state
from classic import *
from quantum_evolve import evolve_quantum
from metrics import *
from plot_utils import *
from config import N, sigma_values, position_seeds, t_values
import os
import matplotlib.pyplot as plt

os.makedirs("results", exist_ok=True)

In [6]:
# 判断 sigma 是否超出周期边界（以中心为准）
def is_sigma_valid(x0, y0, sigma):
    r_max = np.sqrt((np.pi - abs(x0))**2 + (np.pi - abs(y0))**2)
    return 2 * sigma < r_max

# metric

from metrics import *

metrics = {
    "fidelity": compute_fidelity,
    "l2_error": compute_relative_error,
    "density_mse": compute_density_mse,
    "norm_diff": compute_norm_difference,
    "max_density_shift": compute_max_density_shift,
    "spectrum_l2": compute_spectrum_l2_error,
}



# 保存 fidelity vs time 曲线图
fidelity_log = {}

for idx, (x0, y0) in enumerate(position_seeds):
    for sigma in sigma_values:
        if not is_sigma_valid(x0, y0, sigma):
            continue

        psi1_0, psi2_0 = generate_initial_state(x0, y0, sigma)
        initial_state = np.array([psi1_0, psi2_0]).reshape(-1)
        initial_state /= np.linalg.norm(initial_state)

        fidelity_list = []

        for t in t_values:
            psi1_c = evolve_spectral_quantum_style(psi1_0, t, N)
            psi2_c = evolve_spectral_quantum_style(psi2_0, t, N)
            psi1_q, psi2_q = evolve_quantum(5, 5, t, initial_state)

            rho_c = compute_rho(psi1_c, psi2_c)
            rho_q = compute_rho(psi1_q, psi2_q)

            scale = np.sqrt(np.sum(rho_c) / np.sum(rho_q))
            psi1_q *= scale
            psi2_q *= scale


            print(f"Seed {idx} Sigma {sigma} Time {t:.2f} ")
            results = {name: fn(psi1_q, psi2_q, psi1_c, psi2_c) for name, fn in metrics.items()}
            for k, v in results.items():
                if k == "fidelity":
                    print(f"{k.capitalize():20s}: {v:.10f}")
                    # print(f"{k.capitalize():20s}: {v}")
                else:
                    print(f"{k:20s}: {v:.5e}")




Seed 0 Sigma 0.28646831513536897 Time 0.00 
Fidelity            : 1.0000000000
l2_error            : 6.97263e-16
density_mse         : 2.61818e-31
norm_diff           : 1.13687e-13
max_density_shift   : 5.00000e+00
spectrum_l2         : 2.36354e-16
Seed 0 Sigma 0.28646831513536897 Time 0.79 
Fidelity            : 1.0000000000
l2_error            : 1.66294e+00
density_mse         : 9.49170e-31
norm_diff           : 1.13687e-13
max_density_shift   : 0.00000e+00
spectrum_l2         : 4.34433e-16
Seed 0 Sigma 0.28646831513536897 Time 1.57 
Fidelity            : 1.0000000000
l2_error            : 1.84776e+00
density_mse         : 1.19780e-30
norm_diff           : 1.13687e-13
max_density_shift   : 1.60000e+01
spectrum_l2         : 3.35506e-16
Seed 0 Sigma 0.28646831513536897 Time 2.36 
Fidelity            : 1.0000000000
l2_error            : 3.90181e-01
density_mse         : 1.56535e-30
norm_diff           : 1.13687e-13
max_density_shift   : 0.00000e+00
spectrum_l2         : 3.88988e-16
Seed

In [3]:
# === Step 1: 运行主程序后，把每一步的结果收集到 all_results 中 ===

all_results = []

for idx, (x0, y0) in enumerate(position_seeds):
    for sigma in sigma_values:
        if not is_sigma_valid(x0, y0, sigma):
            continue

        psi1_0, psi2_0 = generate_initial_state(x0, y0, sigma)
        initial_state = np.array([psi1_0, psi2_0]).reshape(-1)
        initial_state /= np.linalg.norm(initial_state)

        for t in t_values:
            psi1_c = evolve_rk4_baseline(psi1_0, t, N)
            psi2_c = evolve_rk4_baseline(psi2_0, t, N)
            psi1_q, psi2_q = evolve_quantum(5, 5, t, initial_state)

            # 匹配归一化尺度
            rho_c = compute_rho(psi1_c, psi2_c)
            rho_q = compute_rho(psi1_q, psi2_q)
            scale = np.sqrt(np.sum(rho_c) / np.sum(rho_q))
            psi1_q *= scale
            psi2_q *= scale

            results = {name: fn(psi1_q, psi2_q, psi1_c, psi2_c) for name, fn in metrics.items()}

            row = {
                "seed_idx": idx,
                "x0": x0,
                "y0": y0,
                "sigma": sigma,
                "t": t
            }
            row.update(results)
            all_results.append(row)




In [5]:
# === Step 2: 将 all_results 转为结构化 Excel 表格 ===
import pandas as pd

metric_names = list(metrics.keys())

# 读取成 DataFrame
df_raw = pd.DataFrame(all_results)

# 添加 case_id 标识列
df_raw["case_id"] = df_raw.apply(lambda row: f"x{row.x0:.2f}_y{row.y0:.2f}_s{row.sigma:.2f}", axis=1)

# 添加 step 序号（按 t 排序）
df_raw = df_raw.sort_values(by=["case_id", "t"])
df_raw["step"] = df_raw.groupby("case_id").cumcount() + 1

# 转换为结构化字典：(case_id, metric) -> step -> value
reshaped = {}
for step, group in df_raw.groupby("step"):
    step_key = f"step{int(step)}"
    for _, row in group.iterrows():
        case_id = row["case_id"]
        for metric in metric_names:
            reshaped.setdefault((case_id, metric), {})[step_key] = row[metric]

# 构造 MultiIndex 和 DataFrame
multi_index = pd.MultiIndex.from_tuples(reshaped.keys(), names=["case", "metric"])
df_structured = pd.DataFrame.from_dict(reshaped, orient="index")
df_structured = df_structured.sort_index()
df_structured.columns.name = "step"

# 保存为 Excel 文件
df_structured.to_excel("structured_metrics_output_t=pi_over_2.xlsx")
print("✅ 所有评估结果已保存至 structured_metrics_output.xlsx")


✅ 所有评估结果已保存至 structured_metrics_output.xlsx
